Create Gold Layer Tables

- dim_stores
- dim_family 
- dim_date 
- dim_oil
- dim_holiday
- fact_daily_sales

In [0]:
# Set up paths
catalog = "store_sales_catalog"
raw_path = f"/Volumes/{catalog}/raw/store_sales_vol"
bronze_schema = f"{catalog}.bronze" 
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA gold")

DataFrame[]

In [0]:
%sql
-- Clean Up Tables if needed
DROP TABLE IF EXISTS store_sales_catalog.gold.dim_date;
DROP TABLE IF EXISTS store_sales_catalog.gold.dim_family;
DROP TABLE IF EXISTS store_sales_catalog.gold.dim_holiday;
DROP TABLE IF EXISTS store_sales_catalog.gold.dim_oil;
DROP TABLE IF EXISTS store_sales_catalog.gold.dim_store;
DROP TABLE IF EXISTS store_sales_catalog.gold.fact_daily_sales;

In [0]:
#  GOLD LAYER: STAR SCHEMA + INCREMENTAL STREAMING FACT TABLE

# 1) CREATE GOLD DIMENSION TABLES

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dim_store (
  store_key BIGINT,
  store_nbr INT,
  city STRING,
  state STRING,
  type STRING,
  cluster INT,
  created_at TIMESTAMP,
  PRIMARY KEY (store_key)
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dim_family (
  family_key BIGINT,
  family_name STRING,
  created_at TIMESTAMP,
  PRIMARY KEY (family_key)
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dim_date (
  date_key INT,
  date DATE,
  day_of_week INT,
  week_of_year INT,
  month INT,
  month_name STRING,
  year INT,
  is_weekend BOOLEAN,
  created_at TIMESTAMP,
  PRIMARY KEY (date_key)
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dim_oil (
  oil_key BIGINT,
  date DATE,
  dcoilwtico DOUBLE,
  PRIMARY KEY (oil_key)
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS dim_holiday (
  holiday_key BIGINT,
  date_key INT,
  type STRING,
  locale STRING,
  locale_name STRING,
  description STRING,
  transferred BOOLEAN,
  is_holiday_effective BOOLEAN,
  created_at TIMESTAMP,
  PRIMARY KEY (holiday_key)
) USING DELTA
""")


DataFrame[]

In [0]:
# 2) UPSERT (INITIAL LOAD) OF DIMENSION TABLES FROM SILVER

# Dim Store
spark.sql(f"""
MERGE INTO dim_store tgt
USING {silver_schema}.stores_silver src
ON tgt.store_key = src.store_key
WHEN MATCHED THEN UPDATE SET
  tgt.store_key = src.store_key,
  tgt.store_nbr = src.store_nbr,
  tgt.city = src.city,
  tgt.state = src.state,
  tgt.type = src.type,
  tgt.cluster = src.cluster,
  tgt.created_at = src.created_at
WHEN NOT MATCHED THEN INSERT (
  store_key,
  store_nbr,
  city,
  state,
  type,
  cluster,
  created_at
) VALUES (
  src.store_key,
  src.store_nbr,
  src.city,
  src.state,
  src.type,
  src.cluster,
  src.created_at
)
""")

# Dim Family
spark.sql(f"""
MERGE INTO dim_family tgt
USING {silver_schema}.family_silver src
ON tgt.family_key = src.family_key
WHEN MATCHED THEN UPDATE SET
  tgt.family_key = src.family_key,
  tgt.family_name = src.family_name,
  tgt.created_at = src.created_at
WHEN NOT MATCHED THEN INSERT (
  family_key,
  family_name,
  created_at
) VALUES (
  src.family_key,
  src.family_name,
  src.created_at
)
""")

# Dim Date
spark.sql(f"""
MERGE INTO dim_date tgt
USING {silver_schema}.date_silver src
ON tgt.date_key = src.date_key
WHEN MATCHED THEN UPDATE SET
  tgt.date_key = src.date_key,
  tgt.date = src.date,
  tgt.day_of_week = src.day_of_week,
  tgt.week_of_year = src.week_of_year,
  tgt.month = src.month,
  tgt.month_name = src.month_name,
  tgt.year = src.year,
  tgt.is_weekend = src.is_weekend,
  tgt.created_at = src.created_at
WHEN NOT MATCHED THEN INSERT (
  date_key,
  date,
  day_of_week,
  week_of_year,
  month,
  month_name,
  year,
  is_weekend,
  created_at
) VALUES (
  src.date_key,
  src.date,
  src.day_of_week,
  src.week_of_year,
  src.month,
  src.month_name,
  src.year,
  src.is_weekend,
  src.created_at
)
""")

# Dim Oil
spark.sql(f"""
MERGE INTO dim_oil tgt
USING {silver_schema}.oil_silver src
ON tgt.oil_key = src.oil_key  
WHEN MATCHED THEN UPDATE SET
  tgt.oil_key = src.oil_key,
  tgt.date = src.date,
  tgt.dcoilwtico = src.dcoilwtico
WHEN NOT MATCHED THEN INSERT (
  oil_key,
  date,
  dcoilwtico
) VALUES (
  src.oil_key,
  src.date,
  src.dcoilwtico
)
""")

# Dim Holiday
spark.sql(f"""
MERGE INTO dim_holiday tgt
USING {silver_schema}.holidays_events_silver src
ON tgt.holiday_key = src.holiday_key
WHEN MATCHED THEN UPDATE SET
  tgt.holiday_key = src.holiday_key,
  tgt.type = src.type,
  tgt.locale = src.locale,
  tgt.locale_name = src.locale_name,
  tgt.description = src.description,
  tgt.transferred = src.transferred,
  tgt.created_at = src.created_at
WHEN NOT MATCHED THEN INSERT (
  holiday_key,
  type,
  locale,
  locale_name,
  description,
  transferred,
  created_at
) VALUES (
  src.holiday_key,
  src.type,
  src.locale,
  src.locale_name,
  src.description,
  src.transferred,
  src.created_at
)
""")


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Validate test for DIM tables

In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.dim_date WHERE date_key IS NULL LIMIT 10

date_key,date,day_of_week,week_of_year,month,month_name,year,is_weekend,created_at


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.date_silver WHERE date_key IS NULL LIMIT 10

date_key,date,day_of_week,week_of_year,month,month_name,year,is_weekend,day_of_month,day_name,created_at


In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.dim_family LIMIT 10

family_key,family_name,created_at
1,AUTOMOTIVE,2025-12-04T10:27:16.776Z
2,BABY CARE,2025-12-04T10:27:16.776Z
3,BEAUTY,2025-12-04T10:27:16.776Z
4,BEVERAGES,2025-12-04T10:27:16.776Z
5,BOOKS,2025-12-04T10:27:16.776Z
6,BREAD/BAKERY,2025-12-04T10:27:16.776Z
7,CELEBRATION,2025-12-04T10:27:16.776Z
8,CLEANING,2025-12-04T10:27:16.776Z
9,DAIRY,2025-12-04T10:27:16.776Z
10,DELI,2025-12-04T10:27:16.776Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.dim_holiday WHERE date_key IS NULL LIMIT 10

holiday_key,date_key,type,locale,locale_name,description,transferred,is_holiday_effective,created_at
1,null,Holiday,Local,Manta,Fundacion de Manta,false,null,2025-12-04T10:28:05.402Z
2,null,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,false,null,2025-12-04T10:28:05.402Z
3,null,Holiday,Local,Cuenca,Fundacion de Cuenca,false,null,2025-12-04T10:28:05.402Z
4,null,Holiday,Local,Libertad,Cantonizacion de Libertad,false,null,2025-12-04T10:28:05.402Z
5,null,Holiday,Local,Riobamba,Cantonizacion de Riobamba,false,null,2025-12-04T10:28:05.402Z
6,null,Holiday,Local,Puyo,Cantonizacion del Puyo,false,null,2025-12-04T10:28:05.402Z
7,null,Holiday,Local,Guaranda,Cantonizacion de Guaranda,false,null,2025-12-04T10:28:05.402Z
8,null,Holiday,Local,Latacunga,Cantonizacion de Latacunga,false,null,2025-12-04T10:28:05.402Z
9,null,Holiday,Local,Machala,Fundacion de Machala,false,null,2025-12-04T10:28:05.402Z
10,null,Holiday,Regional,Imbabura,Provincializacion de Imbabura,false,null,2025-12-04T10:28:05.402Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.holidays_events_silver WHERE date_key IS NULL LIMIT 10

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8983520142873323>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * FROM store_sales_catalog.silver.holidays_events_silver WHERE date_key IS NULL LIMIT 10\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:194, in SqlMagic.sql(self, line, cell)
    188 except Ba

In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.dim_oil WHERE oil_key IS NULL LIMIT 10

oil_key,date,dcoilwtico


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.oil_silver WHERE oil_key IS NULL LIMIT 10

oil_key,date,dcoilwtico,created_at,updated_at


In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.dim_store LIMIT 10

store_key,store_nbr,city,state,type,cluster,created_at
1,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z
2,2,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z
3,3,Quito,Pichincha,D,8,2025-12-04T10:27:06.216Z
4,4,Quito,Pichincha,D,9,2025-12-04T10:27:06.216Z
5,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4,2025-12-04T10:27:06.216Z
6,6,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z
7,7,Quito,Pichincha,D,8,2025-12-04T10:27:06.216Z
8,8,Quito,Pichincha,D,8,2025-12-04T10:27:06.216Z
9,9,Quito,Pichincha,B,6,2025-12-04T10:27:06.216Z
10,10,Quito,Pichincha,C,15,2025-12-04T10:27:06.216Z


In [0]:
# 3) CREATE FACT TABLE

spark.sql(f"""
CREATE TABLE IF NOT EXISTS fact_daily_sales (
  sales_fact_id BIGINT GENERATED ALWAYS AS IDENTITY,
  date_key INT,
  store_key BIGINT,
  family_key BIGINT,
  holiday_key BIGINT,
  oil_key BIGINT,

  date DATE,
  sales DOUBLE,
  onpromotion INT,
  transactions INT,
  oil_price DOUBLE,
  is_holiday BOOLEAN,

  created_at TIMESTAMP,
  updated_at TIMESTAMP,

  PRIMARY KEY (sales_fact_id)
)
USING DELTA
PARTITIONED BY (date_key)
""")

DataFrame[]

In [0]:
# 4) STREAMING UPSERT FUNCTION (SILVER -> GOLD FACT TABLE)

def upsert_daily_fact(microBatchDF, batchId):
    microBatchDF.createOrReplaceTempView("silver_batch")

    spark.sql(f"""
        MERGE INTO fact_daily_sales tgt
        USING (
            SELECT
                d.date_key,
                st.store_key,
                f.family_key,
                h.holiday_key,
                o.oil_key,

                s.date,
                s.sales,
                s.onpromotion,
                t.transactions,
                o.dcoilwtico AS oil_price,

                (
                    COALESCE(h.is_national, false)
                    OR COALESCE(h.is_regional, false)
                    OR COALESCE(h.is_local, false)
                ) AS is_holiday,

                current_timestamp() AS created_at,
                current_timestamp() AS updated_at

            FROM silver_batch s
            LEFT JOIN {silver_schema}.date_silver d 
                ON s.date = d.date
            LEFT JOIN {silver_schema}.stores_silver st 
                ON s.store_nbr = st.store_nbr
            LEFT JOIN {silver_schema}.family_silver f 
                ON s.family = f.family_name
            LEFT JOIN {silver_schema}.holidays_events_silver h
                ON s.date = h.date
            LEFT JOIN {silver_schema}.oil_silver o
                ON s.date = o.date
            LEFT JOIN {silver_schema}.transactions_silver t
                ON s.date = t.date AND s.store_nbr = t.store_nbr
        ) src

        ON tgt.date_key   = src.date_key
        AND tgt.store_key = src.store_key
        AND tgt.family_key = src.family_key

        WHEN MATCHED THEN UPDATE SET
            tgt.sales        = src.sales,
            tgt.onpromotion  = src.onpromotion,
            tgt.transactions = src.transactions,
            tgt.oil_price    = src.oil_price,
            tgt.is_holiday   = src.is_holiday,
            tgt.updated_at   = current_timestamp()

        WHEN NOT MATCHED THEN INSERT (
            date_key,
            store_key,
            family_key,
            holiday_key,
            oil_key,
            date,
            sales,
            onpromotion,
            transactions,
            oil_price,
            is_holiday,
            created_at,
            updated_at
        )
        VALUES (
            src.date_key,
            src.store_key,
            src.family_key,
            src.holiday_key,
            src.oil_key,
            src.date,
            src.sales,
            src.onpromotion,
            src.transactions,
            src.oil_price,
            src.is_holiday,
            src.created_at,
            src.updated_at
        )

        """)

In [0]:

# 5) START STREAMING JOB (trigger once)

dbutils.fs.rm(f"{raw_path}/_chk_gold_fact", recurse=True)

chk_gold_path = f"{raw_path}/_chk_gold_fact"

(
    spark.readStream.table(f"{silver_schema}.sales_silver")
        .writeStream
        .trigger(once=True)
        .option("checkpointLocation", chk_gold_path)
        .foreachBatch(upsert_daily_fact)
        .start()
)

print("Gold Fact streaming MERGE job started (trigger once).")


Gold Fact streaming MERGE job started (trigger once).


In [0]:
%sql
SELECT * FROM store_sales_catalog.gold.fact_daily_sales LIMIT 10

sales_fact_id,date_key,store_key,family_key,holiday_key,oil_key,date,sales,onpromotion,transactions,oil_price,is_holiday,created_at,updated_at
86897,20130218,1,8,null,null,2013-02-18,610.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86905,20130218,1,12,null,null,2013-02-18,92.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86913,20130218,1,16,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86921,20130218,1,17,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86929,20130218,1,18,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86937,20130218,1,21,null,null,2013-02-18,1.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86945,20130218,1,26,null,null,2013-02-18,91.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86953,20130218,1,28,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86961,20130218,1,30,null,null,2013-02-18,65.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86969,20130218,10,2,null,null,2013-02-18,0.0,0,1105,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.sales_silver WHERE date = '2013-02-18'

sales_id,date,sales,family,store_nbr,onpromotion,is_promotion,created_at,updated_at
86216,2013-02-18,1799.0,GROCERY I,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86224,2013-02-18,24.0,GROCERY II,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86232,2013-02-18,72.0,"LIQUOR,WINE,BEER",1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86240,2013-02-18,262.232,MEATS,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86248,2013-02-18,0.0,SCHOOL AND OFFICE SUPPLIES,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86256,2013-02-18,21.654,SEAFOOD,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86264,2013-02-18,250.939,MEATS,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86272,2013-02-18,0.0,PET SUPPLIES,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86280,2013-02-18,46.0,EGGS,11,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
86288,2013-02-18,0.0,LAWN AND GARDEN,11,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.oil_silver WHERE date = '2013-02-18'

oil_key,date,dcoilwtico,created_at,updated_at


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.holidays_events_silver WHERE date = '2013-02-18'

holiday_key,date,type,locale,locale_name,description,transferred,is_national,is_regional,is_local,created_at,updated_at


In [0]:
df_fact_daily_sales = spark.table('store_sales_catalog.gold.fact_daily_sales')
df_fact_daily_sales.summary().display()                                  

summary,sales_fact_id,date_key,store_key,family_key,holiday_key,oil_key,sales,onpromotion,transactions,oil_price
count,3054348,3054348,3054348,3054348,502524,2099196,3054348,3054348,2805231,2099196
mean,1527238.1747279617,2.0149062681446906E7,27.5,17.0,184.45744680851064,580.9966044142615,359.0208922513903,2.6174797370830043,1697.0714411754327,68.01587436332568
stddev,881825.2706081857,13411.725086710783,15.585786672418031,9.521906130136705,82.66178685696843,335.0557855692863,1107.2858972351364,12.25493857883017,966.8316523645958,25.69134234718428
min,1,20130101,1,1,42,1,0.0,0,5,26.19
25%,763502,20140301,14,9,113,292,0.0,0,1046,46.41
50%,1527131,20150428,28,17,184,581,11.0,0,1395,53.41
75%,2290772,20160622,41,25,256,870,196.0,0,2081,95.81
max,3099592,20170815,54,33,327,1163,124717.0,741,8359,110.62


In [0]:
%sql
SELECT 
  date, 
  ROUND(SUM(sales), 2) AS sales 
FROM store_sales_catalog.silver.sales_silver 
GROUP BY 1 
ORDER BY 1 ASC

date,sales
2013-01-01,2511.62
2013-01-02,496092.42
2013-01-03,361461.23
2013-01-04,354459.68
2013-01-05,477350.12
2013-01-06,519695.4
2013-01-07,336122.8
2013-01-08,318347.78
2013-01-09,302530.81
2013-01-10,258982.0


In [0]:
%sql
SELECT 
  date, 
  ROUND(SUM(sales), 2) AS sales 
FROM store_sales_catalog.gold.fact_daily_sales 
GROUP BY 1 
ORDER BY 1 ASC

date,sales
2013-01-01,2511.62
2013-01-02,496092.42
2013-01-03,361461.23
2013-01-04,354459.68
2013-01-05,477350.12
2013-01-06,519695.4
2013-01-07,336122.8
2013-01-08,318347.78
2013-01-09,302530.81
2013-01-10,258982.0


In [0]:
%sql
SELECT
    d.date_key,
    st.store_key,
    f.family_key,
    h.holiday_key,
    o.oil_key,
    s.*
FROM store_sales_catalog.silver.sales_silver s
LEFT JOIN store_sales_catalog.silver.date_silver d
    ON s.date = d.date
LEFT JOIN store_sales_catalog.silver.stores_silver st
    ON s.store_nbr = st.store_nbr
LEFT JOIN store_sales_catalog.silver.family_silver f
    ON s.family = f.family_name
LEFT JOIN store_sales_catalog.silver.holidays_events_silver h
    ON s.date = h.date
LEFT JOIN store_sales_catalog.silver.oil_silver o
    ON s.date = o.date
LEFT JOIN store_sales_catalog.silver.transactions_silver t
    ON s.date = t.date AND s.store_nbr = t.store_nbr
LIMIT 50;


date_key,store_key,family_key,holiday_key,oil_key,sales_id,date,sales,family,store_nbr,onpromotion,is_promotion,created_at,updated_at
20130101,1,16,42,null,1,2013-01-01,0.0,HOME AND KITCHEN I,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,1,21,42,null,9,2013-01-01,0.0,LAWN AND GARDEN,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,1,22,42,null,17,2013-01-01,0.0,LINGERIE,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,1,25,42,null,25,2013-01-01,0.0,MEATS,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,1,30,42,null,33,2013-01-01,0.0,PREPARED FOODS,1,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,10,6,42,null,41,2013-01-01,0.0,BREAD/BAKERY,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,10,7,42,null,49,2013-01-01,0.0,CELEBRATION,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,10,20,42,null,57,2013-01-01,0.0,LADIESWEAR,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,10,24,42,null,65,2013-01-01,0.0,MAGAZINES,10,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z
20130101,11,22,42,null,73,2013-01-01,0.0,LINGERIE,11,0,false,2025-12-04T10:27:22.075Z,2025-12-04T10:27:22.075Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.silver.oil_silver WHERE oil_key IS NULL LIMIT 10

oil_key,date,dcoilwtico,created_at,updated_at
